# **ML Translation Model**

## **Import Libraries**

In [ ]:
import numpy as np
from numpy import array
import pandas as pd
import nltk
import re
import tensorflow as tf
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, Bidirectional, Dropout
from sklearn.model_selection import train_test_split
from keras.optimizers import Adam
from keras.preprocessing import sequence
from keras.callbacks import Callback
nltk.download('punkt')

## **Data Collecting and Cleansing**

In [ ]:
df=pd.read_csv("ara_eng.txt", delimiter="\t", names=["english", "arabic"])

In [ ]:
ds = df.loc[0:12000, :].copy()
ds

## **Lowercasing**

In [ ]:
def preprocess(text):
  text = re.sub(r'\b\w+\b',lambda match: match.group(0).lower(),text)
  return text
ds.loc[:, 'english'] = ds['english'].apply(preprocess)
ds['english']

## **Tokenization And Indexing**

In [ ]:
def tokenization(dataset):
  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(dataset)
  return tokenizer

tokenizer=tokenization(ds)

tokenizer_eng=tokenization(ds['english'])
VOCAB_SIZE1=len(tokenizer_eng.word_index) + 1
eng_length=8

word_to_index_english = {word: idx for idx, word in enumerate(tokenizer_eng.word_index)}
index_to_word_english = {index: word for word, index in tokenizer_eng.word_index.items()}

tokenizer_ara=tokenization(ds['arabic'])
VOCAB_SIZE2=len(tokenizer_ara.word_index) + 1
ara_length=8

word_to_index_arabic = {word: idx for idx, word in enumerate(tokenizer_ara.word_index.items())}
index_to_word_arabic = {index: word for word, index in tokenizer_ara.word_index.items()}

print("\nEnglish vocabulary size:",VOCAB_SIZE1)
print("\nArabic vocabulary size:",VOCAB_SIZE2)

## **Encoding and Pad Sequence**

In [ ]:
def encode_sequence(tokenizer, length, ds):
  #integer encode sequences
  seq = tokenizer.texts_to_sequences(ds)
  #pad sequences with 0 value
  seq = pad_sequences(seq, maxlen=length, padding = 'post')
  return seq

## **Splitting the data**

In [ ]:
main_data,test_data,main_label,test_label=train_test_split(ds['english'],ds['arabic'],test_size=0.2,random_state=12)
train_data,val_data,train_label,val_label=train_test_split(main_data,main_label,test_size=0.2,random_state=12)

## **Preparing the data**

In [ ]:
#prepare training data
trainX = encode_sequence(tokenizer_eng, eng_length , train_data)
trainY = encode_sequence(tokenizer_ara , ara_length , train_label)

#prepare validating data
valX = encode_sequence(tokenizer_eng, eng_length, val_data)
valY = encode_sequence(tokenizer_ara, ara_length, val_label)

#prepare test data
testX = encode_sequence(tokenizer_eng, eng_length , test_data)
testY = encode_sequence(tokenizer_ara , ara_length , test_label)

## **Model Building**

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=VOCAB_SIZE1, output_dim=100, input_length=trainX.shape[1]))
model.add(Bidirectional(LSTM(units=64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)))
model.add(Dense(units=VOCAB_SIZE2, activation='sigmoid', use_bias=True, kernel_initializer='glorot_uniform'))

## **Model Compilation and Training**

In [ ]:
class BLEUMetric(Callback):
    def __init__(self, validation_data, index_to_word_english, index_to_word_arabic):
        super(BLEUMetric, self).__init__()
        self.validation_data = validation_data
        self.index_to_word_english = index_to_word_english
        self.index_to_word_arabic = index_to_word_arabic

    def on_epoch_end(self, epoch, logs=None):
        y_true = self.validation_data[1]  # Assuming validation_data is a tuple (X_val, y_val)
        y_pred = self.model.predict(self.validation_data[0])

        bleu_scores = []

        for true_seq, pred_seq in zip(y_true, y_pred):
            # Convert sequences from indices to actual words
            reference_words = [self.index_to_word_english.get(idx, 'UNKNOWN') for idx in true_seq]
            prediction_words = [self.index_to_word_arabic.get(np.argmax(pred_word_probs), 'UNKNOWN') for pred_word_probs in pred_seq]

            # Calculate BLEU score for each pair of reference and prediction
            bleu = nltk.translate.bleu_score.sentence_bleu([reference_words], prediction_words)
            bleu_scores.append(bleu)

        # Calculate the average BLEU score for the entire validation set
        avg_bleu = np.mean(bleu_scores)
        print(f'Epoch {epoch + 1}, Average BLEU Score on Validation Set: {avg_bleu}')

validation_data = (valX, valY)
bleu_metric = BLEUMetric(validation_data, index_to_word_english, index_to_word_arabic)

model.compile(loss='sparse_categorical_crossentropy', optimizer='adamax', metrics=['accuracy'])
history = model.fit(trainX, trainY, epochs=5, batch_size=32, validation_data=(valX, valY), callbacks=[bleu_metric])


## **Evaluation**

In [ ]:
score, acc = model.evaluate(testX, testY, batch_size=32)
print('Test score:', score)
print('Test accuracy:', acc)

## **Prediction**

In [ ]:
english_sentences = ds['english'].head(5)

encoded_english = encode_sequence(tokenizer_eng, eng_length, english_sentences)
encoded_english

predicted_arabic_sequences = model.predict(encoded_english)

for idx, pred_seq in enumerate(predicted_arabic_sequences):
    translated_sentence = ' '.join([index_to_word_arabic.get(idx) for idx in np.argmax(pred_seq, axis=-1) if idx != 0])
    print(f"English Sentence {idx}: {english_sentences.iloc[idx-1]}")
    print(f"Arabic Translation {idx}: {translated_sentence}\n")
    if idx == 5:
        break

## **Transformer**


In [ ]:
!pip install transformers[sentencepiece]
!pip install sentencepiece

!pip install --upgrade transformers

# Use a pipeline as a high-level helper
from transformers import pipeline
import sentencepiece

pipe = pipeline("translation_en_to_ar", model="marefa-nlp/marefa-mt-en-ar")

# Translate text
text = "how are you?"
output = pipe(text)

print(output)